In [ ]:
import os

os.environ['KAGGLE_API_TOKEN'] = "Kaggle API"

In [2]:
!kaggle competitions download -c super-ai-engineer-ss-6-sleep-stage-classification

#!ls

!unzip super-ai-engineer-ss-6-sleep-stage-classification.zip -d super-ai-engineer-ss-6-sleep-stage-classification

#!ls super-ai-engineer-ss-6-sleep-stage-classification

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
  inflating: super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test004/test004_00448.csv  
  inflating: super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test004/test004_00449.csv  
  inflating: super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test004/test004_00450.csv  
  inflating: super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test004/test004_00451.csv  
  inflating: super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test004/test004_00452.csv  
  inflating: super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test004/test004_00453.csv  
  inflating: super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test004/test004_00454.csv  
  inflating: super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test004/test004_00455.csv  
  infla

In [3]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

base_path = '/content/super-ai-engineer-ss-6-sleep-stage-classification'
train_path = os.path.join(base_path, 'train/train')
test_root = os.path.join(base_path, 'test_segment/test_segment')

def extract_signal_features(df):
    f = {}
    cols = ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']

    for c in cols:
        if c in df.columns:
            data = df[c].values
            f[f'{c}_mean'] = np.mean(data)
            f[f'{c}_std'] = np.std(data)
            f[f'{c}_max'] = np.max(data)
            f[f'{c}_min'] = np.min(data)
            f[f'{c}_range'] = f[f'{c}_max'] - f[f'{c}_min']

    if all(c in df.columns for c in ['ACC_X', 'ACC_Y', 'ACC_Z']):
        mag = np.sqrt(df['ACC_X']**2 + df['ACC_Y']**2 + df['ACC_Z']**2)
        f['ACC_mag_mean'] = np.mean(mag)
        f['ACC_mag_std'] = np.std(mag)
        f['ACC_mag_energy'] = np.sum(mag**2)

    return f

train_files = glob.glob(os.path.join(train_path, "*.csv"))
train_data = []

for f in tqdm(train_files):
    df = pd.read_csv(f)
    for i in range(0, len(df), 480):
        chunk = df.iloc[i:i+480]
        if len(chunk) < 400: continue

        feat = extract_signal_features(chunk)
        feat['target'] = chunk['Sleep_Stage'].iloc[0]
        train_data.append(feat)

train_df = pd.DataFrame(train_data).fillna(0)

target_map = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
inv_target_map = {v: k for k, v in target_map.items()}
train_df['label'] = train_df['target'].map(target_map)

X = train_df.drop(columns=['target', 'label'])
y = train_df['label']

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)
model.fit(X, y)

sample_sub = pd.read_csv(os.path.join(base_path, 'sample_submission.csv'))

test_files_map = {os.path.splitext(os.path.basename(f))[0]: f
                  for f in glob.glob(os.path.join(test_root, "**/*.csv"), recursive=True)}

test_preds = []
for sid in tqdm(sample_sub['id']):
    if sid in test_files_map:
        t_df = pd.read_csv(test_files_map[sid])
        feat = extract_signal_features(t_df)

        feat_df = pd.DataFrame([feat]).fillna(0).reindex(columns=X.columns, fill_value=0)

        pred_idx = model.predict(feat_df)[0]
        test_preds.append(inv_target_map[pred_idx])
    else:
        test_preds.append('W')

sample_sub['labels'] = test_preds
sample_sub.to_csv('sleep_ml_statistical.csv', index=False)

100%|██████████| 83/83 [03:16<00:00,  2.37s/it]


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.091245 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10965
[LightGBM] [Info] Number of data points in the train set: 66745, number of used features: 43
[LightGBM] [Info] Start training from score -1.439099
[LightGBM] [Info] Start training from score -2.152800
[LightGBM] [Info] Start training from score -0.680833
[LightGBM] [Info] Start training from score -3.348594
[LightGBM] [Info] Start training from score -2.250266


100%|██████████| 7832/7832 [01:25<00:00, 91.73it/s]


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

base_path = '/content/super-ai-engineer-ss-6-sleep-stage-classification'
train_path = os.path.join(base_path, 'train/train')
test_path = os.path.join(base_path, 'test_segment/test_segment')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cols = ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']
target_map = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
inv_target_map = {v: k for k, v in target_map.items()}

train_files = glob.glob(os.path.join(train_path, "*.csv"))

all_chunks = []
all_labels = []

scaler = StandardScaler()

for f in tqdm(train_files):
    df = pd.read_csv(f)
    df[cols] = df[cols].fillna(0)

    for i in range(0, len(df), 480):
        chunk = df.iloc[i : i+480]
        if len(chunk) < 480: continue

        data = chunk[cols].values
        label = target_map[chunk['Sleep_Stage'].iloc[0]]

        all_chunks.append(data)
        all_labels.append(label)

all_chunks_np = np.array(all_chunks)
num_samples, seq_len, num_features = all_chunks_np.shape
all_chunks_np = all_chunks_np.reshape(-1, num_features)
all_chunks_scaled = scaler.fit_transform(all_chunks_np)
all_chunks_scaled = all_chunks_scaled.reshape(num_samples, seq_len, num_features)

class SleepDataset(Dataset):
    def __init__(self, data, labels=None):
        self.data = torch.tensor(data, dtype=torch.float32).permute(0, 2, 1)
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.data[idx], self.labels[idx]
        return self.data[idx]

train_dataset = SleepDataset(all_chunks_scaled, all_labels)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

class_weights = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

class SleepModel(nn.Module):
    def __init__(self, num_features=8, num_classes=5):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(num_features, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.lstm = nn.LSTM(input_size=64, hidden_size=64, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(64 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x

model = SleepModel().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

epochs = 150
model.train()
for epoch in range(epochs):
    total_loss = 0
    correct = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Accuracy: {acc:.4f}")



100%|██████████| 83/83 [01:31<00:00,  1.10s/it]


Epoch 1/150 | Loss: 1.5074 | Accuracy: 0.2613
Epoch 2/150 | Loss: 1.4191 | Accuracy: 0.2760
Epoch 3/150 | Loss: 1.3474 | Accuracy: 0.3084
Epoch 4/150 | Loss: 1.3460 | Accuracy: 0.3164
Epoch 5/150 | Loss: 1.3005 | Accuracy: 0.3260
Epoch 6/150 | Loss: 1.2595 | Accuracy: 0.3416
Epoch 7/150 | Loss: 1.2379 | Accuracy: 0.3524
Epoch 8/150 | Loss: 1.2173 | Accuracy: 0.3648
Epoch 9/150 | Loss: 1.2394 | Accuracy: 0.3541
Epoch 10/150 | Loss: 1.1908 | Accuracy: 0.3760
Epoch 11/150 | Loss: 1.1730 | Accuracy: 0.3786
Epoch 12/150 | Loss: 1.1633 | Accuracy: 0.3867
Epoch 13/150 | Loss: 1.1674 | Accuracy: 0.3854
Epoch 14/150 | Loss: 1.1540 | Accuracy: 0.3998
Epoch 15/150 | Loss: 1.1407 | Accuracy: 0.4089
Epoch 16/150 | Loss: 1.1280 | Accuracy: 0.4105
Epoch 17/150 | Loss: 1.1180 | Accuracy: 0.4135
Epoch 18/150 | Loss: 1.1103 | Accuracy: 0.4202
Epoch 19/150 | Loss: 1.1040 | Accuracy: 0.4182
Epoch 20/150 | Loss: 1.1062 | Accuracy: 0.4219
Epoch 21/150 | Loss: 1.0865 | Accuracy: 0.4282
Epoch 22/150 | Loss: 1